# Imports

In [1]:
import json
import requests

import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path

# Params

In [2]:
PROJECT_ROOT = Path.cwd().parent

OUTPUT_DIR = PROJECT_ROOT / "data" / "raw"

QUERY_PATH = OUTPUT_DIR / "arxiv_query.json"

In [3]:
with open(
    QUERY_PATH,
    encoding="utf-8",
) as file:
    query = json.load(file)

QUERY = query["query"]

# QUERY = "multi agent systems"

MAX_RESULTS = 10

In [4]:
BASE_URL = "http://export.arxiv.org/api/query"

params = {
    "search_query": f"all:{QUERY}",
    "start": 0,
    "max_results": MAX_RESULTS,
    "sortBy": "submittedDate",
    "sortOrder": "descending",
}

In [5]:
params

{'search_query': 'all:(all:"gaussian distribution" OR all:"gaussian function" OR all:"normal distribution") AND all:"probability theory"',
 'start': 0,
 'max_results': 10,
 'sortBy': 'submittedDate',
 'sortOrder': 'descending'}

# Call API

In [6]:
response = requests.get(
    BASE_URL,
    params=params,
    timeout=30,
)

response.raise_for_status()

print(response.status_code)

200


In [7]:
print(response.text[:1500])

<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/K1y5Q/j2oLU4zlHfpJ3vTRMVR6s</id>
  <title>arXiv Query: search_query=(all:"gaussian distribution" OR all:"gaussian function" OR all:"normal distribution") AND all:"probability theory"&amp;id_list=&amp;start=0&amp;max_results=10</title>
  <updated>2026-07-31T10:31:17Z</updated>
  <link href="https://arxiv.org/api/query?search_query=(all:%22gaussian+distribution%22+OR+(all:%22gaussian+function%22+OR+all:%22normal+distribution%22))+AND+all:%22probability+theory%22&amp;start=0&amp;max_results=10&amp;id_list=" type="application/atom+xml"/>
  <opensearch:itemsPerPage>10</opensearch:itemsPerPage>
  <opensearch:totalResults>30</opensearch:totalResults>
  <opensearch:startIndex>0</opensearch:startIndex>
  <entry>
    <id>http://arxiv.org/abs/2603.21391v2</id>
    <title>A Constructive Appr

In [8]:
root = ET.fromstring(response.text)

In [9]:
namespace = {
    "atom": "http://www.w3.org/2005/Atom"
}

In [10]:
entries = root.findall(
    "atom:entry",
    namespace,
)

len(entries)

10

# Build a table from response

In [11]:
papers = []

for entry in entries:

    title = entry.find(
        "atom:title",
        namespace,
    ).text.strip()

    summary = entry.find(
        "atom:summary",
        namespace,
    ).text.strip()

    published = entry.find(
        "atom:published",
        namespace,
    ).text

    url = entry.find(
        "atom:id",
        namespace,
    ).text

    authors = [
        author.find(
            "atom:name",
            namespace,
        ).text
        for author in entry.findall(
            "atom:author",
            namespace,
        )
    ]

    papers.append(
        {
            "title": title,
            "authors": ", ".join(authors),
            "published": published,
            "summary": summary,
            "url": url,
        }
    )

In [12]:
df = pd.DataFrame(papers)

In [13]:
df

,title,authors,published,summary,url
0,A Constructive Approach to $q$-Gaussian Distri...,"Hiroki Suyari, Antonio M. Scarfone",2026-03-22T20:25:23Z,The Large Deviation Principle (LDP) and the Ce...,http://arxiv.org/abs/2603.21391v2
1,Revisiting De Moivre-Laplace,Raphaël Cerf,2025-12-26T16:28:57Z,We revisit the proof of the de Moivre--Laplace...,http://arxiv.org/abs/2512.22330v1
2,"Topics in Probability, Parametric Estimation a...",Levi Lopes de Lima,2025-10-23T03:24:33Z,We begin our journey by recalling the fundamen...,http://arxiv.org/abs/2510.20163v3
3,The fast rate of convergence of the smooth ada...,"Martin Larsson, Jonghwa Park, Johannes Wiesel",2025-03-13T19:16:19Z,Estimating a $d$-dimensional distribution $μ$ ...,http://arxiv.org/abs/2503.10827v2
4,Probabilistic interpretation of the Selberg--D...,Maximilian Janisch,2025-01-29T10:10:33Z,"In analytic number theory, the Selberg--Delang...",http://arxiv.org/abs/2501.17535v1
5,Bias and Division in the Free World,"Larry Goldstein, Todd Kemp",2024-03-28T22:11:26Z,Sampling bias is a foundational concept in sta...,http://arxiv.org/abs/2403.19860v2
6,Towards a Compositional Framework for Convex A...,"Dario Stein, Richard Samuelson",2023-12-04T19:12:41Z,We introduce a compositional framework for con...,http://arxiv.org/abs/2312.02291v2
7,Gaussian Approximation of Convex Sets by Inter...,"Anindya De, Shivam Nadimpalli, Rocco A. Servedio",2023-11-14T22:42:47Z,We study the approximability of general convex...,http://arxiv.org/abs/2311.08575v1
8,On Polymer Statistical Mechanics: From Gaussia...,Lixiang Yang,2023-08-22T14:54:57Z,Macroscopic mechanical properties of polymers ...,http://arxiv.org/abs/2308.11482v1
9,Diffusion Probabilistic Model Based Accurate a...,"Zezhou Zhang, Chuanchuan Yang, Yifeng Qin, Hao...",2023-04-25T08:25:23Z,Conventional meta-atom designs rely heavily on...,http://arxiv.org/abs/2304.13038v1


# Save results

In [14]:
df.to_csv(
    OUTPUT_DIR / "arxiv_search.csv",
    index=False,
)